In [10]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

In [11]:
df = pd.read_csv('../data/Final Data/Chl-a/Chl-a-7-day.csv')

In [12]:
df

,Chl-a,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A
0,28.47000,0.01470,0.01250,0.05010,0.07250,0.03760,0.04330,0.02510,0.02410,0.02210,0.02060
1,16.77000,0.01410,0.01110,0.05110,0.07140,0.03700,0.03630,0.02280,0.02330,0.01970,0.01820
2,14.30000,0.02110,0.01920,0.05890,0.07830,0.04330,0.04440,0.03060,0.03120,0.02720,0.02720
3,4.64000,0.01520,0.01270,0.04730,0.05320,0.02530,0.02410,0.01920,0.02110,0.01890,0.01760
4,20.48000,0.00550,0.00370,0.05360,0.07130,0.03460,0.03440,0.01780,0.01600,0.01560,0.01300
...,...,...,...,...,...,...,...,...,...,...,...
3815,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565
3816,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565
3817,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565
3818,0.27000,0.00535,0.00355,0.04060,0.04695,0.01710,0.01650,0.01765,0.01765,0.01370,0.01330


In [13]:
df['Chl-a'] = np.log1p(df['Chl-a'])

In [14]:
Q1 = df['Chl-a'].quantile(0.25)
Q3 = df['Chl-a'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df['Chl-a'] >= lower_bound) & (df['Chl-a'] <= upper_bound)]

In [15]:
X = df.drop(columns=['Chl-a'])
y = df['Chl-a']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [17]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
xgb_model = xgb.XGBRegressor(objective="reg:squarederror")

In [19]:
param_dist = {
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 10, 15],
    "n_estimators": [50, 100, 200, 500],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}


In [20]:
tuner = RandomizedSearchCV(xgb_model, param_distributions=param_dist, n_iter=20, scoring='r2', cv=5, verbose=1, random_state=42, n_jobs=-1)
tuner.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None,
                                          random_state=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1.0],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 5, 10, 15],
                                        'n_estimators': [50, 100, 200, 500],
                                        'subsample': [0.6, 0.8, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

In [21]:
best_xgb_model = tuner.best_estimator_
best_xgb_model

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [22]:
y_pred = best_xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Best Parameters: {tuner.best_params_}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R2 Score: {r2}")

Best Parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
MAE: 0.07835460528259569
MSE: 0.015187556916164817
RMSE: 0.12323780635894498
R2 Score: 0.6814216455836124
